In [182]:
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score

# Breast Cancer

In [183]:
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer()
X = cancer.data
y = cancer.target
feature_names = cancer.feature_names

In [184]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [185]:
pipe = Pipeline(
    [("scaler", StandardScaler()), ("lr", LogisticRegression(max_iter=1000))]
)

In [186]:
results = []
num_total_features = X.shape[1]

# Remove one feature at a time, starting from the first feature (index 0)
for num_features_removed in range(num_total_features):
    # Select all features except the first num_features_removed features
    X_subset = X[:, num_features_removed:]

    # Perform cross-validation
    scores = cross_val_score(pipe, X_subset, y, cv=skf)

    results.append(
        {
            # number of features left after removal
            "num_features": X_subset.shape[1],
            "mean_score": np.mean(scores),
            "std_score": np.std(scores),
        }
    )

# Print results
for r in results:
    print(
        f"Features: {r['num_features']:2d}, CV Accuracy: {r['mean_score']:.4f} ± {r['std_score']:.4f}"
    )

Features: 30, CV Accuracy: 0.9737 ± 0.0166
Features: 29, CV Accuracy: 0.9737 ± 0.0166
Features: 28, CV Accuracy: 0.9772 ± 0.0163
Features: 27, CV Accuracy: 0.9772 ± 0.0163
Features: 26, CV Accuracy: 0.9789 ± 0.0131
Features: 25, CV Accuracy: 0.9789 ± 0.0131
Features: 24, CV Accuracy: 0.9754 ± 0.0195
Features: 23, CV Accuracy: 0.9789 ± 0.0163
Features: 22, CV Accuracy: 0.9737 ± 0.0184
Features: 21, CV Accuracy: 0.9772 ± 0.0153
Features: 20, CV Accuracy: 0.9737 ± 0.0222
Features: 19, CV Accuracy: 0.9737 ± 0.0184
Features: 18, CV Accuracy: 0.9737 ± 0.0184
Features: 17, CV Accuracy: 0.9754 ± 0.0195
Features: 16, CV Accuracy: 0.9737 ± 0.0200
Features: 15, CV Accuracy: 0.9737 ± 0.0200
Features: 14, CV Accuracy: 0.9754 ± 0.0231
Features: 13, CV Accuracy: 0.9772 ± 0.0197
Features: 12, CV Accuracy: 0.9772 ± 0.0119
Features: 11, CV Accuracy: 0.9789 ± 0.0119
Features: 10, CV Accuracy: 0.9789 ± 0.0119
Features:  9, CV Accuracy: 0.9789 ± 0.0163
Features:  8, CV Accuracy: 0.9596 ± 0.0142
Features:  

In [187]:
feature_names[[0, 1]]

array(['mean radius', 'mean texture'], dtype='<U23')

In [188]:
results

[{'num_features': 30,
  'mean_score': np.float64(0.9736686849868033),
  'std_score': np.float64(0.016627222216787987)},
 {'num_features': 29,
  'mean_score': np.float64(0.9736686849868033),
  'std_score': np.float64(0.016627222216787987)},
 {'num_features': 28,
  'mean_score': np.float64(0.9771774569166277),
  'std_score': np.float64(0.016256136974859583)},
 {'num_features': 27,
  'mean_score': np.float64(0.9771774569166277),
  'std_score': np.float64(0.016256136974859583)},
 {'num_features': 26,
  'mean_score': np.float64(0.9789318428815401),
  'std_score': np.float64(0.013114128316674255)},
 {'num_features': 25,
  'mean_score': np.float64(0.9789318428815401),
  'std_score': np.float64(0.013114128316674255)},
 {'num_features': 24,
  'mean_score': np.float64(0.9754230709517155),
  'std_score': np.float64(0.019523487856826307)},
 {'num_features': 23,
  'mean_score': np.float64(0.9789318428815401),
  'std_score': np.float64(0.016257812427341933)},
 {'num_features': 22,
  'mean_score': np

### Using RFE with a Pipeline as the Estimator

You *can* pass a pipeline (`Pipeline`) as the `estimator` parameter to `RFE`. For example, if your pipeline includes preprocessing steps like `StandardScaler` followed by a model like `LogisticRegression`:

However, be cautious:

Since RFE treats the entire pipeline as a single estimator, it might face issues accessing attributes like coef_ or feature_importances_ from the final model step. This can cause errors during feature ranking.

Recommended Approach
It's usually more reliable to:

Use RFE with the raw model (e.g., LogisticRegression), not the full pipeline.

Select features based on RFE results.

Then apply the pipeline (scaling + modeling) on the reduced dataset.

This separation avoids potential attribute access issues inside RFE and ensures smoother feature selection and training.

In [189]:
from sklearn.feature_selection import RFE

model = LogisticRegression(max_iter=5000, solver="saga")
rfe = RFE(estimator=model, n_features_to_select=3)
rfe

,estimator,LogisticRegre...solver='saga')
,n_features_to_select,3
,step,1
,verbose,0
,importance_getter,'auto'
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1


In [190]:
scaler = StandardScaler()
X_sc = scaler.fit_transform(X)
rfe.fit_transform(X_sc, y)

print(len(feature_names), len(rfe.ranking_))

30 30


In [191]:
import pandas as pd

ranking_df = pd.DataFrame(
    {"Feature": feature_names, "Ranking": rfe.ranking_}
).sort_values(by="Ranking")

ranking_df

,Feature,Ranking
20,worst radius,1
23,worst area,1
27,worst concave points,1
10,radius error,2
21,worst texture,3
22,worst perimeter,4
7,mean concave points,5
13,area error,6
26,worst concavity,7
15,compactness error,8


In [192]:
rfe.support_

array([False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False,  True, False, False,  True, False, False, False,
        True, False, False])

In [193]:
selected_features = feature_names[rfe.support_]

print("Selected features:")
for feat in selected_features:
    print(feat)

Selected features:
worst radius
worst area
worst concave points


# Wine

In [167]:
from sklearn.datasets import load_wine

data = load_wine()
X = data.data
y = data.target
feature_names = data.feature_names

In [168]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [169]:
pipe = Pipeline(
    [("scaler", StandardScaler()), ("lr", LogisticRegression(max_iter=1000))]
)

In [170]:
results = []
num_total_features = X.shape[1]

# Remove one feature at a time, starting from the first feature (index 0)
for num_features_removed in range(num_total_features):
    # Select all features except the first num_features_removed features
    X_subset = X[:, num_features_removed:]

    # Perform cross-validation
    scores = cross_val_score(pipe, X_subset, y, cv=skf)

    results.append(
        {
            # number of features left after removal
            "num_features": X_subset.shape[1],
            "mean_score": np.mean(scores),
            "std_score": np.std(scores),
        }
    )

# Print results
for r in results:
    print(
        f"Features: {r['num_features']:2d}, CV Accuracy: {r['mean_score']:.4f} ± {r['std_score']:.4f}"
    )

Features: 13, CV Accuracy: 0.9833 ± 0.0136
Features: 12, CV Accuracy: 0.9887 ± 0.0138
Features: 11, CV Accuracy: 0.9775 ± 0.0213
Features: 10, CV Accuracy: 0.9494 ± 0.0379
Features:  9, CV Accuracy: 0.9549 ± 0.0461
Features:  8, CV Accuracy: 0.9549 ± 0.0461
Features:  7, CV Accuracy: 0.9662 ± 0.0331
Features:  6, CV Accuracy: 0.9492 ± 0.0523
Features:  5, CV Accuracy: 0.9492 ± 0.0523
Features:  4, CV Accuracy: 0.9381 ± 0.0415
Features:  3, CV Accuracy: 0.9268 ± 0.0427
Features:  2, CV Accuracy: 0.9098 ± 0.0379
Features:  1, CV Accuracy: 0.6856 ± 0.0313


In [195]:
from sklearn.feature_selection import RFE

model = LogisticRegression(max_iter=5000, solver="saga")
rfe = RFE(estimator=model, n_features_to_select=3)
rfe

,estimator,LogisticRegre...solver='saga')
,n_features_to_select,3
,step,1
,verbose,0
,importance_getter,'auto'
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
